# <div align="center">🪐 مدار (Madar) - النماذج العملاقة 14B على Google Colab</div>
### <div align="center">Frontier-Class 14B Parameter AI Inference with NVIDIA Tesla T4 OpenCL Zero-Copy</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com) 
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar) 
[![Parameters](https://img.shields.io/badge/Model%20Scale-14%20Billion-red.svg)]() 
[![VRAM](https://img.shields.io/badge/GPU%20VRAM-15%20GB%20Tesla%20T4-green.svg)]() 
[![Acceleration](https://img.shields.io/badge/Zero--Copy-OpenCL%20mmap-purple.svg)]()

---

مرحباً بك في دفتر **مدار للنماذج العملاقة (Madar 14B Frontier Edition)**.

تم تخصيص هذا الدفتر لاختبار أقصى ما يمكن لبيئة **Google Colab (النسخة المجانية ببطاقة Tesla T4 15 GB VRAM)** تحمله من نماذج الذكاء الاصطناعي الخارقة:

### 🌟 لماذا نماذج 14B تمثل قفزة نوعية هائلة؟
1. **مضاعفة المعاملات (14.7 Billion Params):** 48 طبقة عصبية كاملة، 5,120 أبعاد تضمين، و 40 رأس انتباه متزامن.
2. **الاستغلال المثالي لكارت Tesla T4:** حجم النموذج بالتكميم Q4_0 هو **~8.54 GB**، مما يعني أنه يستقر بالكامل داخل ذاكرة الـ GPU البالغة 15.36 GB، مع بقاء أكثر من 6 GB فائضاً لـ KV-Cache وسياق المحادثة الممتد!
3. **ذكاء يضاهي النماذج التجارية:** نماذج **Qwen2.5-14B** و **DeepSeek-R1-Distill-14B** تتفوق في المؤشرات العالمية على العديد من النماذج الاحتكارية في الرياضيات والبرمجة واللغة العربية الفصحى.
4. **تسريع Zero-Copy mmap الصافي:** تحميل النموذج الضخم إلى الذاكرة في ثوانٍ معدودة دون نسخ متكرر، وتشغيل طبقات FFN المتجهة عبر OpenCL.

## ⚙️ الخطوة 1: فحص العتاد والذاكرة الرسومية المتاحة (GPU VRAM Check)
نتأكد من توفر بطاقة Tesla T4 بسعة 15 جيجابايت لدعم نموذج 14B بالكامل في الذاكرة:

In [ ]:
# فحص الذاكرة الرسومية VRAM
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

# فحص دعم المعالج لتقنيات AVX2 و FMA
!lscpu | grep -E "Model name|avx2|fma"

## 🔨 الخطوة 2: تثبيت الاعتماديات وبناء نواة مدار ومكتبة القلم AlQalam
نقوم باستنساخ المشروع وترجمة النواة والامتدادات بأقصى درجات التحسين الموجهة لمعالجات x86_64:

In [ ]:
import os, sys

# استنساخ المشروع أو سحب أحدث التحديثات
if not os.path.exists("/content/Madar/language"):
    !git clone https://github.com/azzeddine2017/Madar.git /content/Madar
    %cd /content/Madar
else:
    %cd /content/Madar
    !git pull

# تثبيت حزم OpenCL و GCC و Poppler
!apt-get update -qq && apt-get install -y -qq build-essential g++ ocl-icd-opencl-dev curl libcurl4-openssl-dev poppler-utils

# بناء النواة ومكتبة القلم
!bash tools/build_linux.sh

# التأكد من جاهزية المحرك المتجهي
!bin/ring -v

## 🪄 الخطوة 3: تسجيل الأمر السحري `%%madar` للتنفيذ المباشر

In [ ]:
from IPython.core.magic import register_cell_magic
import subprocess, tempfile, os, sys

MADAR_BIN = os.path.abspath("bin/ring")

@register_cell_magic
def madar(line, cell):
    """IPython Cell Magic لتنفيذ كود مدار وعرض المخرجات مباشرة"""
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(cell)
        tmp_file = f.name
    
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = os.path.abspath("lib") + ":" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        res = subprocess.run([MADAR_BIN, tmp_file], capture_output=True, text=True, env=env, cwd=cwd)
        if res.stdout:
            print(res.stdout, end="")
        if res.stderr:
            print(res.stderr, end="", file=sys.stderr)
    finally:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

ip = get_ipython()
ip.register_magic_function(madar, 'cell', 'ring')
print("✅ تم تسجيل الأمر السحري %%madar بنجاح لتشغيل كود مدار التفاعلي!")

## 🧠 الخطوة 4: تحميل النماذج العملاقة 14B بضغطة زر واحدة (1-Click Downloads)
تصل سرعة التحميل على Google Colab إلى **80 - 120 ميجابايت/ثانية**، مما يعني أن تحميل النموذج العملاق (8.54 GB) يستغرق **أقل من دقيقة ونصف** فقط!

اختر النموذج الذي ترغب بتجربته:

In [ ]:
# 🥇 [الخيار الأول - موصى به بقوة] تحميل Qwen2.5-14B-Instruct (~8.54 GB)
# أعلى مستويات الذكاء في اللغة العربية، المنطق الحسابي، وصياغة الأكواد البرمجية مباشرة
!bash tools/download_colab_models.sh qwen14b

In [ ]:
# 🥈 [الخيار الثاني] تحميل DeepSeek-R1-Distill-Qwen-14B (~8.54 GB)
# أقوى نماذج الاستدلال والتفكير العميق CoT، يحل أصعب المسائل المنطقية والرياضية التنافسية
!bash tools/download_colab_models.sh deepseek14b

## ⚡ الخطوة 5: باقة تجارب النماذج العملاقة 14B (Frontier Benchmark Suite)
سنقوم بتنفيذ 4 اختبارات تثبت الفرق الجوهري في دقة التفكير والاستيعاب بين نماذج 7B ونماذج 14B:

### 🚀 تجربة 5.1: حل المسائل الفيزيائية والرياضية المركبة (Multi-Step Physics & Math)
مسألة مركبة تتضمن تسارعاً وسرعة وزوايا وتحويلاً دقيقاً للوحدات:

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

# التحقق من النموذج المتاح تلقائياً
cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok
if not fexists(cModelPath) cModelPath = "models/qwen2.5-7b-instruct-q4_0.gguf" ok

see f"[MADAR 14B] جارٍ تحميل النموذج العملاق: {cModelPath}" + nl
t0 = clock()
oModel = ai_open(cModelPath)
t1 = clock()
see f"[MADAR 14B] تم تحميل النموذج في: {(t1 - t0) / clockspersecond()} ثانية عبر Zero-Copy mmap!" + nl

# تفعيل كارت الشاشة Tesla T4 وخيوط المعالجة
oModel.setThreads(4)
oModel.setGPU(1)
oModel.setTemperature(0.5)

cPrompt = "حل هذه المسألة بخطوات رياضية دقيقة باللغة العربية: سيارة بدأت حركتها من السكون بتسارع ثابت مقداره 2.5 م/ث² لمدة 12 ثانية، ثم سارت بسرعة منتظمة لمدة 45 ثانية، ثم ضغط السائق على المكابح لتتوقف بتباطؤ مقداره 3 م/ث². احسب: 1) أقصى سرعة بلغتها بالكيلومتر/ساعة. 2) إجمالي المسافة المقطوعة بالأمتار. 3) إجمالي زمن الرحلة كاملاً بالثواني."

see nl + "========================================================" + nl
see "المسألة الفيزيائية المركبة:" + nl
see cPrompt + nl
see "========================================================" + nl

tGen0 = clock()
cAnswer = oModel.generate(cPrompt, 768)
tGen1 = clock()

see oModel.extractAnswer(cAnswer) + nl
see "========================================================" + nl
see f"سرعة التوليد: {oModel.tokensPerSecond()} رمز/ثانية (الزمن: {(tGen1 - tGen0)/clockspersecond()} ثانية)" + nl

### 🧠 تجربة 5.2: الاستدلال المنطقي العميق وحل الألغاز الصعبة (CoT Deep Reasoning)
اختبار النموذج في لغز منطقي معقد يتطلب استنتاجاً متعدد الاحتمالات مع استعراض خطوات التفكير الداخلي `<think>`:

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf" ok

oModel = ai_open(cModelPath)
oModel.setGPU(1)
oModel.setThreads(4)
oModel.setTemperature(0.6)

cLuzzle = "لغز منطقي صعب: يقف ثلاثة حكماء أمام ملك، ولديه 5 قبعات (3 بيضاء و2 سوداء). أغمض الحكماء أعينهم ووضع الملك قبعة على رأس كل منهم وأخفى القبعتين المتبقيتين. عندما فتحوا أعينهم، كل حكيم يرى قبعات الآخرين فقط دون أن يرى قبعته. سأل الملك الأول: هل تعرف لون قبعتك؟ قال: لا. سأل الثاني: هل تعرف لون قبعتك؟ قال: لا. فسأل الحكيم الثالث (وهو أعمى لا يرى شيئاً): هل تعرف لون قبعتك؟ فقال فوراً: نعم! ما هو لون قبعة الحكيم الثالث وكيف استنتج ذلك منطقياً؟"

see "========================================================" + nl
see "اللغز المنطقي:" + nl + cLuzzle + nl
see "========================================================" + nl

t0 = clock()
cFull = oModel.generate(cLuzzle, 1024)
t1 = clock()

cThinking = oModel.extractThinking(cFull)
cAnswer   = oModel.extractAnswer(cFull)

if cThinking != ""
    see "--- [سلسلة التفكير المنطقي العميق - Chain of Thought] ---" + nl
    see cThinking + nl
    see "--------------------------------------------------------" + nl
ok

see nl + "--- [الحل الحتمي المستنتج] ---" + nl
see cAnswer + nl
see "========================================================" + nl
see f"السرعة: {oModel.tokensPerSecond()} رمز/ثانية (الزمن: {(t1 - t0)/clockspersecond()} ثانية)" + nl

### 💻 تجربة 5.3: كتابة نظام برمجي كامل بلغة مدار (Full System Code Synthesis)
طلب بناء فئة متكاملة لإدارة شجرة بحث ثنائية (Binary Search Tree) مع عمليات الإدراج والبحث والترتيب بلغة مدار:

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

oModel = ai_open(cModelPath)
oModel.setGPU(1)
oModel.setThreads(4)
oModel.setTemperature(0.2)

cCodingTask = "اكتب برنامجاً كاملاً ونظيفاً بلغة Ring / Madar يطبق هيكل بيانات شجرة البحث الثنائية (Binary Search Tree - BST):\n1. فئة العقدة Node وفئة الشجرة BST.\n2. دالة insert لإضافة أرقام.\n3. دالة search للبحث عن قيمة.\n4. دالة inorder_traversal لطباعة الأعداد مرتبة تصاعدياً.\nمع كود تجريبي يوضح عملها وتعليقات عربية احترافية."

see "========================================================" + nl
see "مهمة هندسة البرمجيات:" + nl + cCodingTask + nl
see "========================================================" + nl

see oModel.generate(cCodingTask, 700) + nl

### 🌊 تجربة 5.4: البث الحي المباشر للرموز (Real-Time 14B Streaming)
مشاهدة تدفق الأفكار والكلمات لحظة بلحظة مع كارت الشاشة Tesla T4:

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

oModel = ai_open(cModelPath)
oModel.setGPU(1)
oModel.setThreads(4)

cPrompt = "اكتب مقالاً تحليلياً مكثفاً ومؤثراً باللغة العربية الفصحى عن ثورة لغات البرمجة الأصلية (Native Dialects) مثل مدار، وكيف تدمج بين السرعة الحسابية الفائقة للغات الأنظمة C/C++ مع مرونة الذكاء الاصطناعي التوليدي."

see "================ [البث الحي التدفقي المباشر - 14B] ================" + nl
oModel.generateStream(cPrompt, 400)
see nl + "===================================================================" + nl
see f"سرعة التدفق: {oModel.tokensPerSecond()} رمز/ثانية" + nl

## ✒️ الخطوة 6: البحث الهجين واسترجاع المعرفة بالـ 14B عبر AlQalam (C++20 RAG)
استرجاع المعرفة عبر محرك القلم الهجين (دلالي 70% + لفظي 30%) وصياغة إجابات حتمية بأعلى درجات الدقة والاستيعاب اللغوي:

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"
load "libraries/madar_ai/rag.ring"
load "alqalam.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

oTimer = new QalamChronos()
oTimer.reset()

see "[RAG 14B] تهيئة النموذج العملاق لقاعدة المعرفة... " + nl
llm = ai_open(cModelPath)
llm.setGPU(1)
llm.setThreads(4)
store = new VectorStore(llm)

# إضافة وثائق معرفية متقدمة
store.addText("تتميز لغة مدار بدعم الذاكرة المتجهية ومكتبة AlQalam المكتوبة بلغة C++20 والتي تدمج البحث الدلالي مع الفهرس المعكوس QalamIndex.")
store.addText("محرك مدار يستغل تقنية Zero-Copy mmap مع تسريع OpenCL على كروت NVIDIA لتشغيل نماذج 14B العملاقة بكفاءة استثنائية.")
store.addText("حاسوب كوانتم العملاق يعمل بالاعتماد على التشابك الكمي والتراكب الحسابي لمعالجة مليارات الاحتمالات في أجزاء من الثانية.")

cQuestion = "كيف تتيح منظومة مدار تشغيل نماذج 14B الضخمة على كروت الشاشة بكفاءة؟"
see nl + "السؤال المعرفي: " + cQuestion + nl

# البحث الهجين السريع
oTimer.reset()
aResults = store.hybridSearch(cQuestion, 2, 0.7)
see f"[RAG] نتائج البحث الهجين من مكتبة القلم (زمن البحث: {oTimer.elapsed()}):" + nl

for res in aResults
    see f"  - [تطابق: {res[1]}] (دلالي: {res[3]} | لفظي: {res[4]}) {res[2].cText}" + nl
next

best_text = aResults[1][2].cText
see nl + "[RAG 14B] توليد الإجابة الحتمية المدعمة بالوثيقة:" + nl
cPrompt = f"أجب بدقة وبصياغة عربية متقنة اعتماداً على هذا المرجع: {best_text}\nالسؤال: {cQuestion}"
see llm.generate(cPrompt, 350) + nl

## 🤖 الخطوة 7: الوكيل الذاتي الخارق بنموذج 14B (Autonomous Agent with 14B Brain)
الوكيل الذاتي يستغل قوة المعالجة لنموذج 14B لاستدعاء أدوات الحساب الدقيقة وقراءة الذاكرة المتجهية وتقديم إجابات مبرهنة:

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/agent.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

oAgentModel = ai_open(cModelPath)
oAgentModel.setGPU(1)
oAgentModel.setThreads(4)
oAgent = new AIAgent(oAgentModel)
oAgent.remember("مدار هي لغة برمجة وبيئة تشغيل فائقة السرعة مصممة للذكاء الاصطناعي وتطبيقات النظم")
oAgent.remember("يدعم مدار نماذج 14B العملاقة بفضل المعالجة المتجهية AVX2 وتسريع OpenCL المباشر")

see "[AGENT 14B] الذاكرة المتجهية جاهزة! نقوم الآن بسؤال الوكيل:" + nl

cQuery = "احسب ناتج ضرب 8945 في 762 باستخدام أداة الحساب calc، واذكر لي ملخصاً وافياً عن مدار من ذاكرتك"
see nl + "المستخدم > " + cQuery + nl + nl

cResponse = oAgent.query(cQuery)
see nl + "الوكيل الذكي 14B > " + nl + cResponse + nl

## 💬 الخطوة 8: المحادثة الحية التفاعلية مع العقل العملاق 14B (Interactive Chat Widget)
تحدث مع أقوى نموذج ذكاء اصطناعي محلياً عبر واجهة المحادثة الأنيقة المدمجة:

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import subprocess, os, tempfile

chat_history = []

out_area = widgets.Output()
text_input = widgets.Text(
    placeholder='اطرح أي سؤال صعب أو مسألة رياضية أو برمجية على نموذج 14B...',
    layout=widgets.Layout(width='75%')
)
send_btn = widgets.Button(
    description='إرسال 🚀',
    button_style='success',
    layout=widgets.Layout(width='20%')
)

def ask_madar_14b(prompt):
    ring_code = f'''
load "libraries/madar_ai/madar_ai.ring"
cModel = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModel) cModel = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok
model = ai_open(cModel)
model.setGPU(1)
model.setThreads(4)
model.setTemperature(0.6)
model.setMaxTokens(350)
see model.generate("{prompt}", 350)
'''
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(ring_code)
        tmp_name = f.name
    
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = os.path.abspath("lib") + ":" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        res = subprocess.run([os.path.abspath("bin/ring"), tmp_name], capture_output=True, text=True, env=env, cwd=cwd)
        return res.stdout.strip()
    finally:
        if os.path.exists(tmp_name):
            os.remove(tmp_name)

def on_send(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value = ''
    
    with out_area:
        display(HTML(f'<div style="background-color:#e8f5e9; padding:12px; border-radius:10px; margin:6px 0;"><b>🧑 أنت:</b> {user_msg}</div>'))
        display(HTML('<div id="loading" style="color:#2e7d32;"><i>⏳ عقل 14B العملاق يفكر ويحلل...</i></div>'))
    
    bot_msg = ask_madar_14b(user_msg)
    
    with out_area:
        clear_output(wait=True)
        for role, text in chat_history:
            bg = '#e8f5e9' if role == 'user' else '#f3e5f5'
            author = '🧑 أنت' if role == 'user' else '🪐 مدار (14B)'
            display(HTML(f'<div style="background-color:{bg}; padding:12px; border-radius:10px; margin:6px 0;"><b>{author}:</b> {text}</div>'))
        
        display(HTML(f'<div style="background-color:#f3e5f5; padding:12px; border-radius:10px; margin:6px 0;"><b>🪐 مدار (14B):</b> {bot_msg}</div>'))
    
    chat_history.append(('user', user_msg))
    chat_history.append(('bot', bot_msg))

send_btn.on_click(on_send)
text_input.on_submit(on_send)

controls = widgets.HBox([text_input, send_btn])
display(widgets.VBox([out_area, controls]))